# LWSO-YOLO11n — Train trên Kaggle (VisDrone2019)

Notebook clone code trực tiếp từ GitHub thay vì copy thủ công — luôn chạy đúng phiên bản
mới nhất trên `main`. Sửa code thì push lên GitHub rồi rerun/Save Version, không sửa trực
tiếp trong Kaggle. Dataset tự tải qua `gdown` — không cần tạo Kaggle Dataset/Add Input thủ công.

## Chuẩn bị (làm 1 lần)
1. **Settings → Accelerator → GPU P100 hoặc T4x2** — cả 2 đều dùng được, xem `DEVICE` ở cell
   CẤU HÌNH (`"0"` = 1 GPU, `"0,1"` = cả 2 GPU T4 — code đã hỗ trợ multi-GPU/DDP).
2. **Settings → Internet → ON** (bắt buộc để `git clone`, `pip install`, và tải dataset qua `gdown` hoạt động).
3. Nếu repo GitHub là **private**: sửa `GIT_URL` thành `https://<token>@github.com/...` (Personal
   Access Token) hoặc dùng Kaggle Secrets. Repo public thì bỏ qua bước này.

## Thời gian
~6-8 phút/epoch @960 trên P100 (1 GPU) → `EPOCHS=100` vừa khít phiên 12h. T4x2 (`DEVICE="0,1"`)
nhanh hơn nhưng cần đủ VRAM cho cả 2 GPU cùng lúc — nếu OOM, giảm `BATCH` hoặc dùng 1 GPU.
Muốn train dài hơn: sau mỗi phiên **Save Version** để giữ `DAT-Project/runs/`, phiên sau đặt
`RESUME_FROM` trỏ tới `last.pt` (attach output của version trước làm Input).


In [ ]:
# ===================== CẤU HÌNH =====================
GIT_URL  = "https://github.com/trong5nhan6/DAT-Project.git"
REPO_DIR = "/kaggle/working/DAT-Project"

# idea: baseline | lwso | fap — quyết định model_cfg + use_nwd mặc định (xem configs/<idea>.yaml)
IDEA = "lwso"

# ---- MODEL_CFG: chọn 1 trong các option dưới (None = dùng mặc định của configs/<IDEA>.yaml) ----
MODEL_CFG = None
# MODEL_CFG = "yolo11n.pt"                          # baseline YOLO11n gốc, finetune COCO-pretrained (idea baseline)
# MODEL_CFG = "cfg/base-yolo11n.yaml"                # baseline YOLO11n gốc, train from-scratch    (idea baseline)
# MODEL_CFG = "cfg/ablation/yolo11n-p2-nop5.yaml"    # ablation: chỉ +P2/-P5, module gốc 100%       (idea baseline)
# MODEL_CFG = "cfg/lwso-yolo11n.yaml"                # LWSO đầy đủ: 2.40M params, 21.5 GFLOPs@640    (idea lwso)
# MODEL_CFG = "cfg/lwso-yolo11n-lite.yaml"           # LWSO cắt compute mạnh: 1.12M, 12.8 GFLOPs@640 (idea lwso)
# MODEL_CFG = "cfg/lwso-yolo11n-eff.yaml"            # LWSO đề xuất: 0.96M, 12.4 GFLOPs@640, +ECA P2 (idea lwso)
# MODEL_CFG = "cfg/lwso-yolo11s.yaml"                # bản s — teacher cho knowledge distillation    (idea lwso)
# MODEL_CFG = "cfg/fap-yolo11n.yaml"                 # FAP: FreqMix + P2 (giữ P3/P4/P5): 2.13M, 9.0 GFLOPs@640 (idea fap)

DEVICE       = "0"        # "0" = 1 GPU; "0,1" = ca 2 GPU T4 tren Kaggle (DDP, xem README muc DDP)
IMGSZ        = 640
EPOCHS       = 10
BATCH        = 32          # P100 16GB: 8 an toàn, thử 12 nếu còn VRAM. Voi DEVICE="0,1" day la TONG
                           # batch chia cho 2 GPU (ultralytics tu chia) — khong can tu nhan doi.
# RUN_NAME: de None = TU DAT ten theo IDEA/MODEL_CFG (xem cell TRAIN) -- luon phan anh dung
# ban dang chay, khong the con bi lan ten giua cac idea/kien truc nhu truoc. Chi dat gia tri
# rieng o day neu ban muon 1 cai ten co dinh, tu chon (se GHI DE hoan toan logic tu dat).
RUN_NAME     = None
NO_NWD       = False       # True = ép tắt NWD dù configs/<IDEA>.yaml bật (ablation NWD; idea fap vốn đã không dùng NWD)
NWD_RATIO    = 0.5
NWD_CONSTANT = 12.8
TEST_EVERY   = 2          # 0 = tắt; log mAP tập test mỗi N epoch (chỉ giám sát, không chọn best.pt)
RESUME_FROM  = ""         # vd "/kaggle/input/<output-version-truoc>/DAT-Project/runs/detect/<ten-run-cu>/weights/last.pt"

# ---- Chỉ dùng khi IDEA == "fap" (xem cell PRUNE cuối notebook) ----
PRUNE_SPARSITY = 0.3      # tỉ lệ weight bị zero-out toàn cục (LAMP)
PRUNE_PROTECT  = True     # True = M6 (LAMP + bảo vệ P2/P3), False = M5 (LAMP thường, ablation)


In [ ]:
import os

%pip install -q "ultralytics>=8.3,<8.4"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone --depth 1 {GIT_URL} {REPO_DIR}
%cd {REPO_DIR}

!pip install -q -r requirements.txt

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


In [ ]:
# sanity check nhanh trước khi tốn GPU: custom module + patch parse_model có build đúng không
!pytest tests/ -q


In [ ]:
# ============ Tải VisDrone từ Google Drive + tạo data yaml ============
!pip install -q gdown
!mkdir -p datasets
!gdown 1_DTApN9Vwhex4v2m_39aTmNVF4wUA5E- -O datasets/VisDrone.zip
!unzip -q -o datasets/VisDrone.zip -d datasets

import glob
from pathlib import Path

patterns = ["datasets/VisDrone/images/train",
            "datasets/*/images/train",
            "datasets/images/train"]
hits = [p for pat in patterns for p in glob.glob(pat)]
assert hits, "Khong tim thay images/train duoi datasets/ sau khi giai nen — kiem tra lai file zip/Drive ID"
DATA_ROOT = str(Path(sorted(set(hits))[0]).parents[1])
print("Dataset root:", DATA_ROOT)

names = ["pedestrian", "people", "bicycle", "car", "van",
         "truck", "tricycle", "awning-tricycle", "bus", "motor"]
# Tên riêng, không phải data/visdrone.yaml của repo (file đó trỏ path Windows local).
DATA_YAML = "visdrone_kaggle.yaml"
yaml_text = (
    f"path: {DATA_ROOT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(names))
)
Path(DATA_YAML).write_text(yaml_text)
print(yaml_text)


In [ ]:
# ===================== TRAIN =====================
# train.py merge: configs/base.yaml <- configs/{IDEA}.yaml <- các flag dưới đây (chỉ áp
# dụng khi thực sự khác None/False).
from pathlib import Path

model_arg = RESUME_FROM if RESUME_FROM else MODEL_CFG
nwd_flag = "--no-nwd" if NO_NWD else ""

# Ten run: uu tien RUN_NAME neu ban tu dat; khong thi tu suy ra tu MODEL_CFG (doi kien truc,
# vd "lwso-yolo11n-eff") hoac tu IDEA (dung kien truc mac dinh cua idea, vd "lwso"/"baseline"/
# "fap") -- luon bam sat ban dang chay, khong con bi lan giua cac idea/kien truc nhu truoc.
if RUN_NAME:
    ACTUAL_RUN_NAME = RUN_NAME
elif model_arg:
    ACTUAL_RUN_NAME = Path(model_arg).stem
else:
    ACTUAL_RUN_NAME = IDEA

cmd = f"python train.py --idea {IDEA} --data {DATA_YAML} --imgsz {IMGSZ} "
cmd += f"--epochs {EPOCHS} --batch {BATCH} --device {DEVICE} --name {ACTUAL_RUN_NAME} "
cmd += f"--test-every {TEST_EVERY} --nwd-ratio {NWD_RATIO} --nwd-constant {NWD_CONSTANT} {nwd_flag}"
if model_arg:
    cmd += f" --model {model_arg}"

print(cmd)
!{cmd}


In [ ]:
# ===================== VALIDATE + XUAT KET QUA =====================
import shutil
from pathlib import Path

# runs/detect/{ACTUAL_RUN_NAME} co the bi ultralytics doi ten (vd tenrun2) neu thu muc
# da ton tai tu 1 lan chay truoc trong cung session (exist_ok=False mac dinh) —
# lay thu muc khop ACTUAL_RUN_NAME* moi cap nhat gan nhat thay vi hardcode dung ten.
run_dirs = sorted(Path("runs/detect").glob(f"{ACTUAL_RUN_NAME}*"), key=lambda p: p.stat().st_mtime)
assert run_dirs, f"Khong tim thay runs/detect/{ACTUAL_RUN_NAME}* — train da chay xong chua?"
run_dir = run_dirs[-1]
if run_dir.name != ACTUAL_RUN_NAME:
    print(f"[canh bao] dung {run_dir.name} thay vi {ACTUAL_RUN_NAME} (ultralytics da doi ten do trung thu muc)")

val_cmd = (
    f"python val.py --weights {run_dir}/weights/best.pt "
    f"--data {DATA_YAML} --imgsz {IMGSZ} --split val"
)
print(val_cmd)
!{val_cmd}

best = run_dir / "weights" / "best.pt"
shutil.copy(best, f"/kaggle/working/best_{ACTUAL_RUN_NAME}.pt")
print(f"\nDa copy best.pt -> /kaggle/working/best_{ACTUAL_RUN_NAME}.pt (tai ve o tab Output)")


In [ ]:
# ===================== PRUNE (chỉ áp dụng cho IDEA == "fap") =====================
# Semantic-path-aware LAMP pruning (FAP-YOLO12n Sec 4.2) — chạy SAU khi train xong, không
# tự động trong lúc train. Bỏ qua (không làm gì) nếu IDEA khác "fap".
import shutil
from pathlib import Path

if IDEA == "fap":
    protect_flag = "" if PRUNE_PROTECT else "--no-protect"
    prune_cmd = (
        f"python prune.py --weights {run_dir}/weights/best.pt "
        f"--sparsity {PRUNE_SPARSITY} {protect_flag}"
    )
    print(prune_cmd)
    !{prune_cmd}

    pruned = run_dir / "weights" / "best.pruned.pt"
    shutil.copy(pruned, f"/kaggle/working/best_{ACTUAL_RUN_NAME}_pruned.pt")
    print(f"\nDa copy pruned checkpoint -> /kaggle/working/best_{ACTUAL_RUN_NAME}_pruned.pt "
          f"(tai ve o tab Output)")
else:
    print(f"[bo qua] IDEA={IDEA!r} khac 'fap' — prune.py chi ap dung cho idea fap")


## Sau khi train

- **Save Version** (Save & Run All hoặc Quick Save) để giữ `DAT-Project/runs/` + `best_*.pt` trong Output.
- Train tiếp phiên sau: attach Output của version trước làm Input, đặt
  `RESUME_FROM = "/kaggle/input/<slug>/DAT-Project/runs/detect/<ten-run-cu>/weights/last.pt"`
  (`train.py --model` nhận thẳng `.pt` để finetune tiếp — không phải resume optimizer state
  đúng nghĩa, hợp lý vì mỗi phiên Kaggle là một lượt optimizer/cos_lr schedule mới).
- **Tên run tự đặt** (`RUN_NAME = None` mặc định): tự suy ra từ `MODEL_CFG` nếu bạn đổi kiến
  trúc (vd `lwso-yolo11n-eff`), hoặc từ `IDEA` nếu dùng kiến trúc mặc định (vd `lwso`,
  `baseline`, `fap`) — đổi `IDEA`/`MODEL_CFG` là tên thư mục tự đổi theo, **không còn bị lẫn
  tên giữa các idea/kiến trúc khác nhau như trước nữa**. Chỉ cần tự đặt `RUN_NAME` nếu muốn
  1 cái tên cố định riêng (sẽ ghi đè hoàn toàn logic tự đặt).
- Chạy ablation +P2/−P5 (chưa có idea riêng): giữ `IDEA = "baseline"`, đặt
  `MODEL_CFG = "cfg/ablation/yolo11n-p2-nop5.yaml"` — so sánh mAP50 giữa các run để có bảng ablation.
- **`IDEA = "fap"`**: train xong (P2 head + FreqMix, giữ nguyên P3/P4/P5) thì chạy cell PRUNE
  bên dưới (LAMP + bảo vệ đường P2/P3, `PRUNE_SPARSITY`/`PRUNE_PROTECT` ở cell CẤU HÌNH) —
  đây là bước nén riêng, **không** tự động chạy trong lúc train như `lwso`.
- Đánh giá test-dev: sửa cell VALIDATE, đổi `--split val` thành `--split test`.
- Code luôn lấy từ `main` trên GitHub lúc `git clone`/`git pull` ở cell cấu hình — sửa code
  (kể cả `configs/*.yaml`) thì push lên GitHub rồi rerun cell đó (hoặc Save Version mới),
  không sửa trực tiếp trong Kaggle.
